# Brain_Scape — 01 Data Exploration

Explore downloaded OpenNeuro datasets, inspect scan formats, metadata, and quality metrics.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. List Available Scans

Check what sample data we have in `data/samples/` and `data/raw/`.

In [ ]:
data_dir = Path('../data')
samples_dir = data_dir / 'samples'
raw_dir = data_dir / 'raw'

print('=== Sample Datasets ===')
if samples_dir.exists():
    for item in sorted(samples_dir.rglob('*')):
        if item.is_file() and item.suffix in ['.nii', '.gz', '.dcm', '.edf']:
            size_mb = item.stat().st_size / (1024 * 1024)
            print(f'  {item.relative_to(samples_dir)} ({size_mb:.1f} MB)')
else:
    print('  No samples directory found. Run: python scripts/seed_openneuro.py')

print(f'\n=== Raw Data ===')
if raw_dir.exists():
    for job_dir in sorted(raw_dir.iterdir()):
        if job_dir.is_dir():
            print(f'  Job: {job_dir.name}')
            for f in job_dir.iterdir():
                size_mb = f.stat().st_size / (1024 * 1024)
                print(f'    {f.name} ({size_mb:.1f} MB)')
else:
    print('  No raw data found. Run: python scripts/ingest.py <scan_path>')

## 2. Load and Inspect a NIfTI Scan

Load the first available `.nii.gz` file and examine its properties.

In [ ]:
def find_nifti_files(base_dir):
    """Recursively find all NIfTI files."""
    nii_files = []
    for ext in ['*.nii.gz', '*.nii']:
        nii_files.extend(Path(base_dir).rglob(ext))
    return sorted(nii_files)

nii_files = find_nifti_files(data_dir)

if nii_files:
    scan_path = nii_files[0]
    print(f'Loading: {scan_path}')
    img = nib.load(str(scan_path))
    data = img.get_fdata()
    
    print(f'\n=== Scan Metadata ===')
    print(f'Shape:      {img.shape}')
    print(f'Data type:  {img.get_data_dtype()}')
    print(f'Voxel size: {img.header.get_zooms()}')
    print(f'Affine:\n{img.affine}')
    print(f'\n=== Intensity Statistics ===')
    print(f'Min:  {data.min():.2f}')
    print(f'Max:  {data.max():.2f}')
    print(f'Mean: {data.mean():.2f}')
    print(f'Std:  {data.std():.2f}')
else:
    print('No NIfTI files found. Download sample data first.')

## 3. Visualize Orthogonal Slices

Display axial, coronal, and sagittal views through the center of the volume.

In [ ]:
if nii_files:
    mid_x = data.shape[0] // 2
    mid_y = data.shape[1] // 2
    mid_z = data.shape[2] // 2

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Sagittal
    axes[0].imshow(data[mid_x, :, :].T, cmap='gray', origin='lower')
    axes[0].set_title(f'Sagittal (x={mid_x})')
    axes[0].axis('off')

    # Coronal
    axes[1].imshow(data[:, mid_y, :].T, cmap='gray', origin='lower')
    axes[1].set_title(f'Coronal (y={mid_y})')
    axes[1].axis('off')

    # Axial
    axes[2].imshow(data[:, :, mid_z].T, cmap='gray', origin='lower')
    axes[2].set_title(f'Axial (z={mid_z})')
    axes[2].axis('off')

    plt.suptitle(f'Orthogonal Slices — {scan_path.name}', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No scan loaded. Download sample data first.')

## 4. Intensity Histogram

Plot the intensity distribution, useful for checking normalization needs.

In [ ]:
if nii_files:
    # Use non-zero voxels only (exclude background)
    non_zero = data[data > 0]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].hist(non_zero.ravel(), bins=200, color='steelblue', edgecolor='none')
    axes[0].set_title('Non-zero Intensity Distribution')
    axes[0].set_xlabel('Intensity')
    axes[0].set_ylabel('Voxel Count')

    # Log scale
    axes[1].hist(non_zero.ravel(), bins=200, color='steelblue', edgecolor='none', log=True)
    axes[1].set_title('Non-zero Intensity Distribution (log scale)')
    axes[1].set_xlabel('Intensity')
    axes[1].set_ylabel('Voxel Count (log)')

    plt.tight_layout()
    plt.show()

    # Percentile stats
    pcts = [1, 5, 25, 50, 75, 95, 99]
    print('\nPercentile statistics (non-zero voxels):')
    for p in pcts:
        print(f'  P{p:02d}: {np.percentile(non_zero, p):.2f}')

## 5. Signal-to-Noise Ratio

Estimate SNR using the mean/std method on non-zero voxels.

In [ ]:
if nii_files:
    # Simple SNR estimate: mean(non-zero) / std(non-zero)
    # More sophisticated: SNR = mean(signal) / std(noise)
    # where noise is estimated from background
    background = data[data == 0]
    foreground = data[data > 0]

    if len(background) > 0 and len(foreground) > 0:
        snr_simple = foreground.mean() / foreground.std()
        snr_background = foreground.mean() / background.std()
        print(f'SNR (mean/sigma of foreground):  {snr_simple:.2f}')
        print(f'SNR (mean_foreground/sigma_bg):  {snr_background:.2f}')
        print(f'Foreground mean: {foreground.mean():.2f}')
        print(f'Foreground std:  {foreground.std():.2f}')
        print(f'Background std:  {background.std():.2f}')
    else:
        print('Could not compute SNR — check data.')

## 6. Format Detection

Test the ingestion format detector on sample files.

In [ ]:
from ingestion.format_detector import detect_format

# Test on all found files
for f in nii_files[:5]:  # Limit to first 5
    try:
        fmt, metadata = detect_format(str(f))
        print(f'{f.name}: {fmt.value} | {metadata}')
    except Exception as e:
        print(f'{f.name}: ERROR — {e}')

## 7. Summary & Next Steps

- Verify that scans load correctly and have reasonable dimensions
- Check SNR — if too low, the scan may need extra denoising
- Confirm format detection works for all input types
- Proceed to `02_preprocessing.ipynb` to clean and register the data